# Bhoonidhi SDK — archive

Browse the portal's satellite/sensor catalogue from Python. No login
needed, so this works on a fresh client. Every method matches a `bhd
archive` command. Runs live against the portal.

## 1. Create a client

In [1]:
from bhoonidhi_downloader.sdk import BhoonidhiClient

client = BhoonidhiClient()
client

## 2. List the archive

`client.archive.list()` matches `bhd archive list`. Returns the raw satellite records.

In [2]:
records = client.archive.list()
print("satellites:", len(records))
for r in records[:5]:
    print(" -", r.get("satName"), "|", ", ".join(s.get("senName") for s in r.get("sensors", [])))

satellites: 41
 - Aqua | MODIS
 - CartoSat-2 | PAN(SPOT)
 - CartoSat-2S | MX(SPOT), PAN(SPOT)
 - CartoSat-3 | MX(SPOT), MX(SPOT), PAN(SPOT)
 - EOS-04 | SAR(CRS), SAR(FRS1), SAR(FRS2), SAR(MRS), SAR(MRS), SAR(MRS), SAR(MRS)


## 3. Inspect one satellite

In [3]:
aqua = next(r for r in records if r.get("satName") == "Aqua")
print("satellite:", aqua.get("satName"))
print("access:   ", aqua.get("priced"))
for s in aqua.get("sensors", []):
    print("  sensor:", s.get("senName"), "| res:", s.get("res"), "| products:", s.get("products"))

satellite: Aqua
access:    OpenData_OnOrder
  sensor: MODIS | res: 500 | products: Others


## 4. Export to JSON

`client.archive.export(path)` matches `bhd archive export -o path`. Writes the parsed archive and returns the same records.

In [4]:
import json, tempfile, os

path = os.path.join(tempfile.gettempdir(), "archive_full.json")
parsed = client.archive.export(path)

print("wrote:", path, "(", os.path.getsize(path), "bytes )")
print("records:", len(parsed))
print("first parsed record keys:", sorted(parsed[0].keys()))

wrote: /tmp/archive_full.json ( 55849 bytes )
records: 41
first parsed record keys: ['access_level', 'availability', 'availability_end', 'availability_start', 'collections', 'index', 'max_resolution', 'min_resolution', 'resolution', 'satellite']


## 5. Export a single satellite

In [5]:
path_aqua = os.path.join(tempfile.gettempdir(), "archive_aqua.json")
parsed_aqua = client.archive.export(path_aqua, sat="Aqua")

print("wrote:", path_aqua, "(", os.path.getsize(path_aqua), "bytes )")
print("records:", len(parsed_aqua))
print(json.dumps(parsed_aqua[0], indent=2)[:400])

wrote: /tmp/archive_aqua.json ( 624 bytes )
records: 1
{
  "index": 1,
  "satellite": "Aqua",
  "availability": "31 December 2003 - 31 December 2019",
  "availability_start": "31 December 2003",
  "availability_end": "31 December 2019",
  "access_level": "OnOrder",
  "collections": [
    {
      "Aqua_MODIS": {
        "sensor": "MODIS",
        "resolution": "500",
        "start_date": "12/31/2003",
        "end_date": "12/31/2019",
        "product": "Others",
        "product_token": "",
        "sat_value": "Aqua:MODIS"
      }
    }
  ],
  "re

## 6. Refresh from the portal

`refresh=True` re-fetches instead of using the local cache.

In [6]:
fresh = client.archive.list(refresh=True)
print("re-fetched satellites:", len(fresh))

re-fetched satellites: 41
